# Kumo Tabular on raw series

NVIDIA Kumo Tabular (https://arxiv.org/abs/2611.23001), a tabular in-context-learning model, fed the raw time series: one column per time step, no features. Large model, 8 estimators, resample 0 of the 20 small UCR datasets used for the HC2 run. MultiRocketHydra (aeon 1.6.0, `tscbench_runs`) as reference.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sdm
import torch
from aeon.datasets import load_from_ts_file

DATA = Path("../data")
MRH = Path("../tscbench_runs/MultiRocketHydraClassifier-1.6.0-eada72/Predictions")
DATASETS = [
    "DiatomSizeReduction", "InsectEPGSmallTrain", "BeetleFly", "BirdChicken", "Chinatown",
    "MoteStrain", "Rock", "ShapeletSim", "SonyAIBORobotSurface1", "ECGFiveDays", "TwoLeadECG",
    "FaceFour", "Symbols", "SonyAIBORobotSurface2", "Coffee", "FreezerSmallTrain", "BME",
    "Beef", "CBF", "OliveOil",
]

model = sdm.models.KumoTabular(size="large", device="cuda")


def to_table(X, y=None):
    df = pd.DataFrame(X[:, 0, :], columns=[f"t{i}" for i in range(X.shape[2])])
    if y is not None:
        df["target"] = y
    stypes = sdm.infer_stypes(df, overrides={"target": "categorical"} if y is not None else None)
    return sdm.TableTensor.from_pandas(df=df, stypes=stypes, device="cuda")

In [2]:
rows = []
for ds in DATASETS:
    X_train, y_train = load_from_ts_file(str(DATA / ds / f"{ds}0_TRAIN.ts"))
    X_test, y_test = load_from_ts_file(str(DATA / ds / f"{ds}0_TEST.ts"))
    train, test = to_table(X_train, y_train), to_table(X_test)
    start = time.time()
    with torch.amp.autocast("cuda", torch.float16):
        model.fit(x=train.drop_columns("target"), y=train[:, "target"], num_estimators=8)
        probs = model.predict(test).to_pandas()
    seconds = time.time() - start
    model.clear()
    # output columns are the class labels, in no fixed order
    preds = probs.columns.to_numpy()[probs.to_numpy().argmax(axis=1)]
    mrh = float(open(MRH / ds / "testResample0.csv").readlines()[2].split(",")[0])
    rows.append({"dataset": ds, "length": X_train.shape[2], "n_train": len(y_train),
                 "kumo": (preds == y_test).mean(), "mrh": mrh, "seconds": seconds})
    print(rows[-1])

res = pd.DataFrame(rows).set_index("dataset")
res.loc["mean"] = res.mean()
res.round(4)

/export/home/gpetelin/TSCbench/.venv/lib/python3.13/site-packages/sdm/tensor/string.py:502: RuntimeWarning: Falling back to a CPU-based string sort because cuDF is not installed. Install cuDF to enable faster CUDA-based string sorting without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(


{'dataset': 'DiatomSizeReduction', 'length': 345, 'n_train': 16, 'kumo': np.float64(0.9836601307189542), 'mrh': 0.9640522875816994, 'seconds': 13.018290519714355}


{'dataset': 'InsectEPGSmallTrain', 'length': 601, 'n_train': 17, 'kumo': np.float64(1.0), 'mrh': 1.0, 'seconds': 2.1249752044677734}


{'dataset': 'BeetleFly', 'length': 512, 'n_train': 20, 'kumo': np.float64(0.85), 'mrh': 0.9, 'seconds': 2.072139024734497}


{'dataset': 'BirdChicken', 'length': 512, 'n_train': 20, 'kumo': np.float64(0.85), 'mrh': 0.9, 'seconds': 1.5632898807525635}


{'dataset': 'Chinatown', 'length': 24, 'n_train': 20, 'kumo': np.float64(0.9854227405247813), 'mrh': 0.9795918367346939, 'seconds': 1.2805709838867188}


... (12 outputs left out)

{'dataset': 'Beef', 'length': 470, 'n_train': 30, 'kumo': np.float64(0.8333333333333334), 'mrh': 0.7666666666666667, 'seconds': 1.5195941925048828}


{'dataset': 'CBF', 'length': 128, 'n_train': 30, 'kumo': np.float64(0.9577777777777777), 'mrh': 0.9955555555555555, 'seconds': 1.2983858585357666}


{'dataset': 'OliveOil', 'length': 570, 'n_train': 30, 'kumo': np.float64(0.8666666666666667), 'mrh': 0.9666666666666667, 'seconds': 1.596653938293457}


,length,n_train,kumo,mrh,seconds
dataset,,,,,
DiatomSizeReduction,345.0,16.00,0.9837,0.9641,13.0183
InsectEPGSmallTrain,601.0,17.00,1.0000,1.0000,2.1250
BeetleFly,512.0,20.00,0.8500,0.9000,2.0721
BirdChicken,512.0,20.00,0.8500,0.9000,1.5633
Chinatown,24.0,20.00,0.9854,0.9796,1.2806
MoteStrain,84.0,20.00,0.8922,0.9457,2.2249
Rock,2844.0,20.00,0.7600,0.8600,1.6788
ShapeletSim,500.0,20.00,0.4833,1.0000,2.4576
SonyAIBORobotSurface1,70.0,20.00,0.8719,0.8869,1.6424


In [3]:
MRH_ALL = pd.read_csv("../generated_results/tscbench/Accuracy/all_resamples/MultiRocketHydra_accuracy.csv",
                      index_col=0)["0"]
MORE = [
    "ArrowHead", "Car", "ECG200", "GunPoint", "Ham", "Herring", "Lightning7", "Meat", "Plane",
    "Wine", "Worms", "Trace", "ItalyPowerDemand", "SyntheticControl", "Adiac", "SwedishLeaf",
    "Strawberry", "Earthquakes", "Computers", "Yoga", "Wafer", "FordA",
]

rows = []
for ds in MORE:
    X_train, y_train = load_from_ts_file(str(DATA / ds / f"{ds}0_TRAIN.ts"))
    X_test, y_test = load_from_ts_file(str(DATA / ds / f"{ds}0_TEST.ts"))
    train, test = to_table(X_train, y_train), to_table(X_test)
    start = time.time()
    with torch.amp.autocast("cuda", torch.float16):
        model.fit(x=train.drop_columns("target"), y=train[:, "target"], num_estimators=8)
        probs = model.predict(test).to_pandas()
    seconds = time.time() - start
    model.clear()
    preds = probs.columns.to_numpy()[probs.to_numpy().argmax(axis=1)]
    rows.append({"dataset": ds, "length": X_train.shape[2], "n_train": len(y_train),
                 "kumo": (preds == y_test).mean(), "mrh": MRH_ALL[ds], "seconds": seconds})
    print(rows[-1])

more = pd.DataFrame(rows).set_index("dataset")
more.loc["mean"] = more.mean()
more.round(4)

{'dataset': 'ArrowHead', 'length': 251, 'n_train': 36, 'kumo': np.float64(0.7371428571428571), 'mrh': np.float64(0.8628571428571429), 'seconds': 1.3270392417907715}


{'dataset': 'Car', 'length': 577, 'n_train': 60, 'kumo': np.float64(0.8333333333333334), 'mrh': np.float64(0.9333333333333332), 'seconds': 1.572707176208496}


{'dataset': 'ECG200', 'length': 96, 'n_train': 100, 'kumo': np.float64(0.88), 'mrh': np.float64(0.91), 'seconds': 1.0895030498504639}


{'dataset': 'GunPoint', 'length': 150, 'n_train': 50, 'kumo': np.float64(0.9733333333333334), 'mrh': np.float64(1.0), 'seconds': 1.1296486854553223}


{'dataset': 'Ham', 'length': 431, 'n_train': 109, 'kumo': np.float64(0.7238095238095238), 'mrh': np.float64(0.7428571428571429), 'seconds': 1.455580711364746}


{'dataset': 'Herring', 'length': 512, 'n_train': 64, 'kumo': np.float64(0.546875), 'mrh': np.float64(0.734375), 'seconds': 1.63785719871521}


... (13 outputs left out)

{'dataset': 'Yoga', 'length': 426, 'n_train': 300, 'kumo': np.float64(0.8766666666666667), 'mrh': np.float64(0.929), 'seconds': 6.892480373382568}


{'dataset': 'Wafer', 'length': 152, 'n_train': 1000, 'kumo': np.float64(0.9951330304996755), 'mrh': np.float64(0.999675535366645), 'seconds': 5.377421617507935}


{'dataset': 'FordA', 'length': 500, 'n_train': 3601, 'kumo': np.float64(0.8962121212121212), 'mrh': np.float64(0.9553030303030304), 'seconds': 15.440730094909668}


,length,n_train,kumo,mrh,seconds
dataset,,,,,
ArrowHead,251.0000,36.0000,0.7371,0.8629,1.3270
Car,577.0000,60.0000,0.8333,0.9333,1.5727
ECG200,96.0000,100.0000,0.8800,0.9100,1.0895
GunPoint,150.0000,50.0000,0.9733,1.0000,1.1296
Ham,431.0000,109.0000,0.7238,0.7429,1.4556
Herring,512.0000,64.0000,0.5469,0.7344,1.6379
Lightning7,319.0000,70.0000,0.7808,0.8219,1.5830
Meat,448.0000,60.0000,1.0000,0.9333,1.5796
Plane,144.0000,105.0000,1.0000,1.0000,1.9358
